In [2]:
"""
Script 2 — Validation Dataset (True Binders vs Easy Negatives)
==============================================================
Dataset: merged_mcmahon_data.csv
  - No mutation scanning — samples are real antibodies (binders)
    or easy negatives (target-shuffled / unrelated)
  - Single interface (A,B) — no interface filtering needed
  - Ground truth: 'binder' column (1/0) or 'binding' column (True/False)
  - Extra metadata: KD[M], EC50[M], source, vhh, target, etc.
 
Steps:
  1.  Load data + normalise ground truth
  2.  Parse array-valued Chai columns
  3.  NaN audit + class imbalance report
  4.  Select score columns + auto-direction via ROC-AUC
  5.  Benchmarking: ROC-AUC, PR-AUC, F1, Precision@Opt → rankings.csv
  6.  Visualisations (grouped per model):
        - KDE distributions (binder vs non-binder)
        - Strip + box separability plots
        - Quadrant plots (TP/FP/TN/FN) with sample labels
        - Top-metrics bar chart (PR-AUC + ROC-AUC)
        - PR curves for top-5 metrics
        - Confusion matrix for best metric
"""
 
import re, ast, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, f1_score,
    precision_recall_curve, confusion_matrix,
    auc as auc_fn,
)

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")

# ══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════

# NOTE: Update this path to your actual file location
INPUT_CSV    = "/home/bri/pymol/merged_mcmahon_data.csv" 
OUTPUT_DIR   = Path("mcmahon_test")
OUTPUT_DIR.mkdir(exist_ok=True)

MIN_SAMPLES  = 5    
MODEL_PREFIXES = ["af3", "cf", "esmfold", "chai", "boltz", "pred"]

BINDER_COLOR    = "#2196F3"
NONBINDER_COLOR = "#FF5722"
STRIP_PALETTE   = {"binder": BINDER_COLOR, "non-binder": NONBINDER_COLOR}

META_COLS = {
    "sample", "binding", "binder", "binding_target", "interface",
    "source", "type", "vhh", "vhh_sequence", "target",
    "target_residues", "target_sequence", "notes",
    "KD[M]", "EC50[M]",
    "af3_rank", "af3_rank0_masif", "af3_ranking_score",
    "cf_rank",  "cf_rank0_masif",
}

# ══════════════════════════════════════════════════════════════════
# STEP 1 — Load Data & Normalise Ground Truth
# ══════════════════════════════════════════════════════════════════

print("=" * 65)
print("STEP 1 — Load Data")
print("=" * 65)

df = pd.read_csv(INPUT_CSV)
print(f"Shape: {df.shape}")

if "binding" in df.columns:
    df["binding"] = df["binding"].map(
        {True: True, False: False, "True": True, "False": False,
         1: True, 0: False, "1": True, "0": False})
elif "binder" in df.columns:
    df["binding"] = df["binder"].astype(bool)
else:
    raise ValueError("Need 'binding' (True/False) or 'binder' (1/0) column.")

df["binding_target"] = df["binding"].astype(int)

print(f"Binding distribution:\n{df['binding'].value_counts().to_string()}")

# ══════════════════════════════════════════════════════════════════
# STEP 2 — Parse Array-Valued Chai Columns
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("STEP 2 — Parse Array-Valued Columns")
print("=" * 65)

def flatten_list(l):
    """Robust flattening replacement for deprecated pd.core.common.flatten"""
    for el in l:
        if isinstance(el, (list, tuple, np.ndarray)):
            yield from flatten_list(el)
        else:
            yield el

def parse_array_column(series):
    """Extracts max value from stringified nested lists like '[[0.9, 0.8]]'"""
    out = []
    for val in series:
        if pd.isna(val):
            out.append(np.nan); continue
        try:
            lst  = ast.literal_eval(str(val))
            flat = list(flatten_list(lst))
            nums = [float(x) for x in flat if x is not None]
            out.append(max(nums) if nums else np.nan)
        except (ValueError, SyntaxError, TypeError):
            out.append(np.nan)
    return out

array_cols = [c for c in df.columns if df[c].dtype == "object" 
              and df[c].astype(str).str.contains(r"\[\[", na=False).any()]

if array_cols:
    print(f"Found {len(array_cols)} array-valued columns — extracting max:")
    for col in array_cols:
        print(f"  {col}")
        df[col] = parse_array_column(df[col])

# ══════════════════════════════════════════════════════════════════
# STEP 4 — Select Score Columns + Auto-Direction via ROC-AUC
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("STEP 4 — Select Score Columns + Auto-Direction")
print("=" * 65)

path_cols  = {c for c in df.columns if "path" in c.lower()}
EXCLUDE    = META_COLS | path_cols

score_cols = [
    c for c in df.columns
    if c not in EXCLUDE
    and pd.api.types.is_numeric_dtype(df[c])
    and df[c].nunique() > 1
]

model_groups = {p: [c for c in score_cols if c.startswith(p)] for p in MODEL_PREFIXES}

y_all = df["binding_target"]
DIRECTIONS = {}
dir_rows   = []
has_both = y_all.nunique() > 1

for col in score_cols:
    s = pd.to_numeric(df[col], errors="coerce")
    mask = s.notna() & y_all.notna()
    if not has_both or mask.sum() < MIN_SAMPLES or y_all[mask].nunique() < 2:
        DIRECTIONS[col] = 1
        dir_rows.append({"col": col, "raw_roc": np.nan, "direction": 1, "note": "insufficient_data"})
        continue
    
    raw_roc = roc_auc_score(y_all[mask], s[mask])
    direction = 1 if raw_roc >= 0.5 else -1
    DIRECTIONS[col] = direction
    dir_rows.append({"col": col, "raw_roc": round(raw_roc, 4), 
                     "direction": direction, "note": "flipped" if direction == -1 else "kept"})

# Apply directions
df_aligned = df.copy()
for col in score_cols:
    if DIRECTIONS[col] == -1:
        df_aligned[col] = df_aligned[col] * -1

# ══════════════════════════════════════════════════════════════════
# STEP 5 — Benchmarking
# ══════════════════════════════════════════════════════════════════

results = []
y_true = df_aligned["binding_target"].values

if has_both:
    for col in score_cols:
        s = pd.to_numeric(df_aligned[col], errors="coerce")
        clean_idx = s.notna()
        if clean_idx.sum() < MIN_SAMPLES: continue
        
        y, scores = y_true[clean_idx], s[clean_idx].values
        if len(np.unique(y)) < 2: continue
        
        roc = roc_auc_score(y, scores)
        pr = average_precision_score(y, scores)
        
        pre, rec, thr = precision_recall_curve(y, scores)
        f1s = (2 * pre[:-1] * rec[:-1]) / (pre[:-1] + rec[:-1] + 1e-10)
        best_idx = int(np.argmax(f1s))
        best_thr = float(thr[best_idx])
        
        results.append({
            "Metric": col, "Direction": DIRECTIONS[col],
            "ROC_AUC": round(roc, 4), "PR_AUC": round(pr, 4),
            "F1 (Opt)": round(f1_score(y, (scores >= best_thr)), 4),
            "Opt_Threshold": round(best_thr * DIRECTIONS[col], 4),
        })

results_df = pd.DataFrame(results).sort_values("PR_AUC", ascending=False)
results_df.to_csv(OUTPUT_DIR / "rankings.csv", index=False)

# ══════════════════════════════════════════════════════════════════
# STEP 6 — Visualisations (The Fix)
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("STEP 6 — Visualisations")
print("=" * 65)

for model, cols in model_groups.items():
    valid = [c for c in cols if df_aligned[c].notna().any()]
    if not valid: continue

    # FIXED: Using sns.kdeplot instead of vals.plot.kde
    nc, nr = 3, (len(valid) + 2) // 3
    fig, axes = plt.subplots(nr, nc, figsize=(15, 4 * nr))
    axes = np.atleast_1d(axes).flatten()
    
    for i, col in enumerate(valid):
        ax = axes[i]
        for val, color in [(True, BINDER_COLOR), (False, NONBINDER_COLOR)]:
            vals = df_aligned.loc[df_aligned["binding"] == val, col].dropna()
            if len(vals) > 1:
                # FIX: Seaborn handles 'fill' correctly
                sns.kdeplot(data=vals, ax=ax, color=color, 
                            label="binder" if val else "non-binder",
                            linewidth=2, fill=True, alpha=0.25)
            elif len(vals) == 1:
                ax.axvline(vals.iloc[0], color=color, linewidth=2, 
                           label="binder" if val else "non-binder")
        ax.set_title(col, fontsize=9, fontweight="bold")
        ax.legend(fontsize=7)
        
    for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"03_{model}_distributions.png", dpi=150)
    plt.close()

print(f"\nDONE — results saved in {OUTPUT_DIR}")

STEP 1 — Load Data
Shape: (34, 81)
Binding distribution:
binding
True     18
False    16

STEP 2 — Parse Array-Valued Columns

STEP 4 — Select Score Columns + Auto-Direction

STEP 6 — Visualisations

DONE — results saved in mcmahon_test
